# Prepare Data and populate Mongo Collections

The objective of this notebook is to prepare the data in order to populate our Mongo collections. Those collections will be queried by our Agent later on.

In [1]:
import pandas as pd
import numpy as np
from pymongo import MongoClient
from dotenv import dotenv_values

env_path = "/workspace/.env"
config = dotenv_values(env_path)

In [2]:
client = MongoClient(config['MONGO_CONNECTION_STRING'])
db = client["stylistai"]
print(db.list_collection_names()) # should be empty on first run

['customers']


## Customers

In [3]:
customers_csv = "/data/customers.csv"
customers_df = pd.read_csv(customers_csv)
customers_df.head(3)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...


In [4]:
customers_df.shape

(1371980, 7)

In [5]:
rows_count = customers_df.shape[0]
dashline = "---" * 15

for column in customers_df.columns:
    empty_rows = customers_df[column].isna().sum()
    print(f"{column} column has {empty_rows} NA rows. It represents {np.round((empty_rows / rows_count) * 100, 2)}% of total data.")
    print(dashline)

customer_id column has 0 NA rows. It represents 0.0% of total data.
---------------------------------------------
FN column has 895050 NA rows. It represents 65.24% of total data.
---------------------------------------------
Active column has 907576 NA rows. It represents 66.15% of total data.
---------------------------------------------
club_member_status column has 6062 NA rows. It represents 0.44% of total data.
---------------------------------------------
fashion_news_frequency column has 16011 NA rows. It represents 1.17% of total data.
---------------------------------------------
age column has 15861 NA rows. It represents 1.16% of total data.
---------------------------------------------
postal_code column has 0 NA rows. It represents 0.0% of total data.
---------------------------------------------


In [6]:
cat_columns = ['FN', 'Active', 'club_member_status', 'fashion_news_frequency']

for column in cat_columns:
    print(f"Unique values for {column}: {customers_df[column].unique()}")
    print(dashline)

Unique values for FN: [nan  1.]
---------------------------------------------
Unique values for Active: [nan  1.]
---------------------------------------------
Unique values for club_member_status: ['ACTIVE' nan 'PRE-CREATE' 'LEFT CLUB']
---------------------------------------------
Unique values for fashion_news_frequency: ['NONE' 'Regularly' nan 'Monthly']
---------------------------------------------


In [7]:
customers_df['club_member_status'].value_counts()

club_member_status
ACTIVE        1272491
PRE-CREATE      92960
LEFT CLUB         467
Name: count, dtype: int64

In [8]:
customers_df['fashion_news_frequency'].value_counts()

fashion_news_frequency
NONE         877711
Regularly    477416
Monthly         842
Name: count, dtype: int64

Based on above information:
- 'FN' column should be dropped as too many values are empty and we don't have any information on that column
- we will input 0 to all NaN values for the 'Active' column and assume it flags if the customer bought an article in the last 3 months
- less common value will be inputed to 'club_member_status' and 'club_member_status' empty rows
- 'postal_code' values are encoded and of no use. However, we can replace with random real postal codes 

In [9]:
average_age = int(customers_df['age'].mean())
customers_df.drop(columns=['FN'], inplace=True)
customers_df['Active'].fillna(0, inplace=True)
customers_df['club_member_status'].fillna('LEFT CLUB', inplace=True)
customers_df['fashion_news_frequency'].fillna('Monthly', inplace=True)
customers_df['age'].fillna(average_age, inplace=True)
customers_df['age'] = customers_df['age'].apply(lambda x: int(x))

/tmp/ipykernel_843/853390588.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  customers_df['Active'].fillna(0, inplace=True)
/tmp/ipykernel_843/853390588.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

## Transactions

In [10]:
transactions_csv = "/data/transactions_train.csv"
transactions_df = pd.read_csv(transactions_csv)
transactions_df.head(3)

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2


In [11]:
transactions_df.shape

(31788324, 5)

In [12]:
transactions_df.isna().sum()

t_dat               0
customer_id         0
article_id          0
price               0
sales_channel_id    0
dtype: int64

In [13]:
transactions_df['customer_id'].nunique()

1362281

In [14]:
from tqdm import tqdm

tqdm.pandas()

articles_per_customer = (transactions_df.groupby('customer_id')['article_id']
                        .progress_apply(list)
                        .reset_index(name='purchased_articles'))
print("Done grouping purchases by customer id")
print(dashline)
print("Merging with customers df...")
customers_df = customers_df.merge(articles_per_customer, on='customer_id', how='left')
print("Done!")
print(dashline)
customers_df['purchased_articles'] = customers_df['purchased_articles'].progress_apply(lambda x: x if isinstance(x, list) else [])

100%|██████████| 1362281/1362281 [00:08<00:00, 165960.93it/s]


Done grouping purchases by customer id
---------------------------------------------
Merging with customers df...
Done!
---------------------------------------------


100%|██████████| 1371980/1371980 [00:00<00:00, 4067501.07it/s]


In [15]:
print(f"Counting channels by customers")
channel_counts = (transactions_df.groupby('customer_id')['sales_channel_id']
                .value_counts()
                .reset_index(name='count')
                )

print(dashline)
print("Now getting most used channel for each customer")
most_used_channel = (channel_counts.sort_values(['customer_id', 'count'], ascending=[True, False])
                    .drop_duplicates('customer_id')
                    .rename(columns={'sales_channel_id': 'favorite_sales_channel'})
                    [['customer_id', 'favorite_sales_channel']]
                    )
print(dashline)

print("Merging dataframes...")
customers_df = customers_df.merge(most_used_channel, on='customer_id', how='left')
print("Done! Ready for review.")

Counting channels by customers
---------------------------------------------
Now getting most used channel for each customer
---------------------------------------------
Merging dataframes...
Done! Ready for review.


In [16]:
customers_df.head(3)

,customer_id,Active,club_member_status,fashion_news_frequency,age,postal_code,purchased_articles,favorite_sales_channel
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,"[625548001, 176209023, 627759010, 697138006, 5...",2.0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0.0,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,"[583558001, 639677008, 640244003, 521269001, 6...",2.0
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.0,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,"[663713001, 541518023, 663713001, 578020002, 7...",2.0


## Stores

In [17]:
stores_csv = "/data/HM_all_stores.csv"
stores_df = pd.read_csv(stores_csv)
stores_df.head(3)

,storeCode,storeClass,name,phone,city,country,countryCode,longitude,latitude,timeZoneIndex,...,Tue_open_hours,Wed_open_hours,Thu_open_hours,Fri_open_hours,Sat_open_hours,Sun_open_hours,streetName1,streetName2,state,address_string
0,AE0122,Red,Mirdiff city center,+971-42316646,Dubai,United Arab Emirates,AE,55.424840,25.226280,165.0,...,10:00-22:00,10:00-22:00,10:00-23:55,10:00-23:55,10:00-23:55,10:00-22:00,Mirdiff city center,Sheikh Mohammad Bin Zayed Road,Dubai,Mirdiff city center;Sheikh Mohammad Bin Zayed ...
1,AE0149,Flagship,Dubai Mall,+971-44190346,Dubai,United Arab Emirates,AE,55.278446,25.197506,165.0,...,10:00-23:00,10:00-23:00,10:00-23:55,10:00-23:55,10:00-23:00,10:00-23:00,Dubai Mall,Sheikh Zayed Road,Dubai,Dubai Mall;Sheikh Zayed Road;Dubai;Dubai;Dubai
2,AE0209,Blue,Al Markaziyah,+971-26120870,Abu Dhabi,United Arab Emirates,AE,54.357462,24.487245,165.0,...,10:00-22:00,10:00-22:00,10:00-23:00,10:00-23:00,10:00-22:00,10:00-22:00,Al Markaziyah,World Trade Center Mall,Abu Dhabi,Al Markaziyah;World Trade Center Mall;123;Abu ...


In [18]:
stores_df.tail(3)

,storeCode,storeClass,name,phone,city,country,countryCode,longitude,latitude,timeZoneIndex,...,Tue_open_hours,Wed_open_hours,Thu_open_hours,Fri_open_hours,Sat_open_hours,Sun_open_hours,streetName1,streetName2,state,address_string
4289,ZA0030,Red,Cresta Shopping Centre,+27-860690707,Johannesburg,South Africa,ZA,27.983647,-26.127327,140.0,...,09:00-18:00,09:00-18:00,09:00-18:00,09:00-18:00,09:00-18:00,09:00-17:00,Cnr Beyers Naude Drive and Weltevreden Road,"Cresta Ext4, Randburg",NaN,Cnr Beyers Naude Drive and Weltevreden Road;Cr...
4290,ZA0032,Blue,Eikestad Mall,+27-860690707,Cape Town,South Africa,ZA,18.859995,-33.934854,140.0,...,09:00-18:00,09:00-18:00,09:00-18:00,09:00-18:00,09:00-17:00,09:00-14:00,43 Andringa St,NaN,Western Cape,43 Andringa St;;7600;Stellenbosch Central;West...
4291,ZA0040,Blue,Westgate Shopping Centre,+27-860690707,Johannesburg,South Africa,ZA,27.861970,-26.137210,140.0,...,09:00-18:00,09:00-18:00,09:00-18:00,09:00-18:00,09:00-17:00,09:00-16:00,"Shop no: OOB-1, 120 Ontdekkers Road",Roodepoort,Gauteng,"Shop no: OOB-1, 120 Ontdekkers Road;Roodepoort..."


In [19]:
stores_df.columns

Index(['storeCode', 'storeClass', 'name', 'phone', 'city', 'country',
       'countryCode', 'longitude', 'latitude', 'timeZoneIndex',
       'Mon_open_hours', 'Tue_open_hours', 'Wed_open_hours', 'Thu_open_hours',
       'Fri_open_hours', 'Sat_open_hours', 'Sun_open_hours', 'streetName1',
       'streetName2', 'state', 'address_string'],
      dtype='object')

In [20]:
stores_df.shape

(4292, 21)

In [21]:
stores_df['storeCode'].nunique()

4290

In [22]:
stores_df.drop_duplicates(subset=['storeCode'], inplace=True)

In [23]:
import googlemaps
# https://github.com/googlemaps/google-maps-services-python

GOOGLE_MAPS_API = config['GOOGLE_MAPS_API']

gmaps = googlemaps.Client(key=GOOGLE_MAPS_API)

In [24]:
import time
from googlemaps.exceptions import TransportError, Timeout, ApiError

cached_codes = {}

def get_stores_postal_code(latitude: float, longitude: float, max_retries=5):
    """
    Query Google Maps API to get postal code from latitude/longitude.
    Returns None if no postal code exists for that location.
    """
    
    for attempt in range(max_retries):
        try:
            time.sleep(0.5)  # simple rate limiting
            reverse_geocode_result = gmaps.reverse_geocode((latitude, longitude))
            if not reverse_geocode_result:
                return None

            for component in reverse_geocode_result[0]["address_components"]:
                if "postal_code" in component["types"]:
                    return component["long_name"]
                
            return None
        
        except (TransportError, Timeout) as e:
            time.sleep(2 ** attempt)

        except ApiError as e:
            return None
        
    return None

def cached_get_postal_code(latitude, longitude, cached_codes: dict = cached_codes):
    """
    Cache postal codes to reduce API calls for nearby coordinates.
    """
    
    key = (round(latitude, 5), round(longitude, 5))
    if key in cached_codes:
        return cached_codes[key]

    postal_code = get_stores_postal_code(latitude, longitude)
    cached_codes[key] = postal_code
    
    return postal_code


In [25]:
stores_df["postal_code"] = stores_df.progress_apply(lambda r: cached_get_postal_code(r["latitude"], r["longitude"]),axis=1)

100%|██████████| 4290/4290 [43:12<00:00,  1.65it/s]


In [26]:
stores_df.tail(3)

,storeCode,storeClass,name,phone,city,country,countryCode,longitude,latitude,timeZoneIndex,...,Wed_open_hours,Thu_open_hours,Fri_open_hours,Sat_open_hours,Sun_open_hours,streetName1,streetName2,state,address_string,postal_code
4289,ZA0030,Red,Cresta Shopping Centre,+27-860690707,Johannesburg,South Africa,ZA,27.983647,-26.127327,140.0,...,09:00-18:00,09:00-18:00,09:00-18:00,09:00-18:00,09:00-17:00,Cnr Beyers Naude Drive and Weltevreden Road,"Cresta Ext4, Randburg",NaN,Cnr Beyers Naude Drive and Weltevreden Road;Cr...,2118
4290,ZA0032,Blue,Eikestad Mall,+27-860690707,Cape Town,South Africa,ZA,18.859995,-33.934854,140.0,...,09:00-18:00,09:00-18:00,09:00-18:00,09:00-17:00,09:00-14:00,43 Andringa St,NaN,Western Cape,43 Andringa St;;7600;Stellenbosch Central;West...,7600
4291,ZA0040,Blue,Westgate Shopping Centre,+27-860690707,Johannesburg,South Africa,ZA,27.861970,-26.137210,140.0,...,09:00-18:00,09:00-18:00,09:00-18:00,09:00-17:00,09:00-16:00,"Shop no: OOB-1, 120 Ontdekkers Road",Roodepoort,Gauteng,"Shop no: OOB-1, 120 Ontdekkers Road;Roodepoort...",1724


In [27]:
all_post_codes = list(stores_df['postal_code'].unique())
all_post_codes.remove(None)

In [28]:
customers_df.head(3)

,customer_id,Active,club_member_status,fashion_news_frequency,age,postal_code,purchased_articles,favorite_sales_channel
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,"[625548001, 176209023, 627759010, 697138006, 5...",2.0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0.0,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,"[583558001, 639677008, 640244003, 521269001, 6...",2.0
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.0,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,"[663713001, 541518023, 663713001, 578020002, 7...",2.0


In [29]:
customers_df["postal_code"] = np.random.choice(all_post_codes,size=len(customers_df), replace=True)

In [30]:
customers_df.head(3)

,customer_id,Active,club_member_status,fashion_news_frequency,age,postal_code,purchased_articles,favorite_sales_channel
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0.0,ACTIVE,NONE,49,63450,"[625548001, 176209023, 627759010, 697138006, 5...",2.0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0.0,ACTIVE,NONE,25,08753,"[583558001, 639677008, 640244003, 521269001, 6...",2.0
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0.0,ACTIVE,NONE,24,2068,"[663713001, 541518023, 663713001, 578020002, 7...",2.0


In [31]:
customers_df.to_csv("/data/processed/customers_df", index=False)

## Create Customers Collection

Highly advise you run the script at `/workspace/scripts/populate_mongo/customers.py` instead of the below cell to ensure your process doesn't crash.

In [1]:
import pandas as pd
import numpy as np
from pymongo import MongoClient, UpdateOne
from dotenv import dotenv_values

env_path = "/workspace/.env"
config = dotenv_values(env_path)
client = MongoClient(config['MONGO_CONNECTION_STRING'])
db = client["stylistai"]

In [2]:
customers_df = pd.read_csv("/data/processed/customers_df")

In [3]:
import math

def row_to_doc(row: pd.Series) -> dict:
    """
    Convert a dataframe row to a Mongo ready document.
    """
    
    doc = row.to_dict()

    # Convert NaN to None
    for k, v in list(doc.items()):
        if isinstance(v, float) and math.isnan(v):
            doc[k] = None

    # Ensure purchased_articles is a plain Python list
    pa = doc.get("purchased_articles")
    if pa is None or (isinstance(pa, float) and math.isnan(pa)):
        doc["purchased_articles"] = []
    elif not isinstance(pa, list):
        doc["purchased_articles"] = list(pa)

    return doc


In [4]:
from tqdm import tqdm

dashline = "---" * 15

print("Initiating collection in Mongo")
customers_collection = db["customers"]
customers_collection.create_index("customer_id", unique=True)
print(dashline)

print("Now turning dataframe rows into documents")
docs = [row_to_doc(r) for _, r in tqdm(customers_df.iterrows(), total=len(customers_df))]
print(dashline)

print("Upserting document into collection")
operation = [UpdateOne({"customer_id": d["customer_id"]}, {"$set": d}, upsert=True) for d in tqdm(docs, total=len(docs))]
result = customers_collection.bulk_write(operation, ordered=False)

print({"matched": result.matched_count,
    "modified": result.modified_count,
    "upserted": len(result.upserted_ids) if result.upserted_ids else 0
    })
print(dashline)

Initiating collection in Mongo
---------------------------------------------
Now turning dataframe rows into documents


100%|██████████| 1371980/1371980 [00:26<00:00, 51060.29it/s]


---------------------------------------------
Upserting document into collection


100%|██████████| 1371980/1371980 [00:07<00:00, 193262.59it/s]


: 

## Articles

In [21]:
import pandas as pd

articles_csv = "/data/articles.csv"
articles_df = pd.read_csv(articles_csv)
articles_df.head(3)

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.


In [22]:
articles_df.shape

(105542, 25)

In [23]:
articles_df.columns

Index(['article_id', 'product_code', 'prod_name', 'product_type_no',
       'product_type_name', 'product_group_name', 'graphical_appearance_no',
       'graphical_appearance_name', 'colour_group_code', 'colour_group_name',
       'perceived_colour_value_id', 'perceived_colour_value_name',
       'perceived_colour_master_id', 'perceived_colour_master_name',
       'department_no', 'department_name', 'index_code', 'index_name',
       'index_group_no', 'index_group_name', 'section_no', 'section_name',
       'garment_group_no', 'garment_group_name', 'detail_desc'],
      dtype='object')

In [24]:
for column in articles_df.columns:
    print(f"Value for column: {column}")
    print(articles_df[column][0])
    print("---"*15)

Value for column: article_id
108775015
---------------------------------------------
Value for column: product_code
108775
---------------------------------------------
Value for column: prod_name
Strap top
---------------------------------------------
Value for column: product_type_no
253
---------------------------------------------
Value for column: product_type_name
Vest top
---------------------------------------------
Value for column: product_group_name
Garment Upper body
---------------------------------------------
Value for column: graphical_appearance_no
1010016
---------------------------------------------
Value for column: graphical_appearance_name
Solid
---------------------------------------------
Value for column: colour_group_code
9
---------------------------------------------
Value for column: colour_group_name
Black
---------------------------------------------
Value for column: perceived_colour_value_id
4
---------------------------------------------
Value for colu

Below are the columns to keep for now to create a brief document for each article and insert it in Mongo. 

Script to create the collection available at `/workspace/scripts/populate_mongo/articles.py`.

In [25]:
# Value for column: article_id
# 108775015
# ---------------------------------------------
# Value for column: prod_name
# Strap top
# ---------------------------------------------
# Value for column: product_type_name
# Vest top
# ---------------------------------------------
# Value for column: product_group_name
# Garment Upper body
# ---------------------------------------------
# Value for column: department_name
# Jersey Basic
# ---------------------------------------------
# Value for column: index_name
# Ladieswear
# ---------------------------------------------
# Value for column: section_name
# Womens Everyday Basics
# ---------------------------------------------
# Value for column: detail_desc
# Jersey top with narrow shoulder straps.
# ---------------------------------------------

In [26]:
columns_to_keep = ['article_id', 'prod_name', 'product_type_name', 'product_group_name', 'department_name',
                'index_name', 'section_name', 'detail_desc']
columns_to_delete = list(set(articles_df.columns.to_list()) - set(columns_to_keep))

articles_df.drop(columns=columns_to_delete, inplace=True)

In [27]:
articles_df.head(5)

,article_id,prod_name,product_type_name,product_group_name,department_name,index_name,section_name,detail_desc
0,108775015,Strap top,Vest top,Garment Upper body,Jersey Basic,Ladieswear,Womens Everyday Basics,Jersey top with narrow shoulder straps.
1,108775044,Strap top,Vest top,Garment Upper body,Jersey Basic,Ladieswear,Womens Everyday Basics,Jersey top with narrow shoulder straps.
2,108775051,Strap top (1),Vest top,Garment Upper body,Jersey Basic,Ladieswear,Womens Everyday Basics,Jersey top with narrow shoulder straps.
3,110065001,OP T-shirt (Idro),Bra,Underwear,Clean Lingerie,Lingeries/Tights,Womens Lingerie,"Microfibre T-shirt bra with underwired, moulde..."
4,110065002,OP T-shirt (Idro),Bra,Underwear,Clean Lingerie,Lingeries/Tights,Womens Lingerie,"Microfibre T-shirt bra with underwired, moulde..."


In [28]:
articles_df.to_csv('/data/processed/articles_df', index=False)